# Sprint 6 — 02 Model Registration

**Objective:** log the validated `Civ4RagModel` PyFunc to MLflow and register it in the **Unity Catalog Model Registry**, producing a versioned, traceable deployment artifact.

## Decisions

* **Unity Catalog Model Registry** (registry URI `databricks-uc`) — supported in this workspace (verified: the registry lists `system.ai.*` models). The workspace-registry fallback is unnecessary.
* **Naming convention:** `workspace.gcivilization.civ4_rag_model` — same catalog/schema as all project tables (`civ4_*` convention). The model is created **once**; re-running this notebook appends a **new version** rather than a duplicate model.
* **Traceability:** the registered version points to an MLflow run tagged with the Sprint 5 evaluation evidence — config `B_top10_prompt_v1`, eval run ID, dataset version `v1-1fd1d9e2`, and the summary metrics that justified Config B — so the deployment can be traced back to the evaluation results.
* **Same packaging as notebook 01's validated round trip:** `code_paths` (both modules), signature, input example, and identical `pip_requirements`.

## Inputs / outputs

* **Inputs:** `06_model_serving/rag_model.py`, `04_generation/rag_utils.py` (read-only), `workspace.gcivilization.civ4_evaluation_summary` (Sprint 5 evidence).
* **Outputs:** MLflow run in experiment `/Users/endoerner@gmail.com/civ4-rag-model-serving` + registered model version `workspace.gcivilization.civ4_rag_model` (version 1 on first run).

## Prerequisites

`01_model_packaging` must have passed (validated PyFunc + round-trip artifact).

In [0]:
# Setup — imports, registry/experiment configuration, and the model signature.
import sys
for _dir in (
    "/Workspace/Repos/endoerner@gmail.com/civ4-rag/04_generation",
    "/Workspace/Repos/endoerner@gmail.com/civ4-rag/06_model_serving",
):
    if _dir not in sys.path:
        sys.path.insert(0, _dir)

import json
import mlflow
import pandas as pd
import rag_model
import rag_utils
from mlflow.models import ModelSignature

# Unity Catalog Model Registry
CATALOG = "workspace"
SCHEMA = "gcivilization"
REGISTERED_MODEL_NAME = f"{CATALOG}.{SCHEMA}.civ4_rag_model"
mlflow.set_registry_uri("databricks-uc")

# Dedicated MLflow experiment for the model-serving sprint
EXPERIMENT_NAME = "/Users/endoerner@gmail.com/civ4-rag-model-serving"
mlflow.set_experiment(EXPERIMENT_NAME)

# Signature — identical contract to notebook 01's validated round trip
SIG = ModelSignature(rag_model.INPUT_SCHEMA, rag_model.OUTPUT_SCHEMA)

# Dependencies — MUST stay identical to 01_model_packaging's save_model call
PIP_REQUIREMENTS = ["mlflow>=3.0", "databricks-sdk>=0.122.0", "pandas>=2.0"]

print(f"Registry target : {REGISTERED_MODEL_NAME} (databricks-uc)")
print(f"Experiment      : {EXPERIMENT_NAME}")
print(f"Pinned config   : {rag_model.PINNED_CONFIG['config_name']}")

In [0]:
# Sprint 5 evidence — the registered model must be traceable to the evaluation.
# Read Config B's row from the evaluation summary (the best validated config).
eval_row = spark.sql("""
    SELECT config_name, top_k, prompt_version, temperature,
           generation_model, embedding_model, eval_dataset_version,
           mlflow_run_id, faithfulness, citation_correctness, answer_correctness
    FROM workspace.gcivilization.civ4_evaluation_summary
    WHERE config_name = 'B_top10_prompt_v1'
""").collect()[0]

print("Sprint 5 evidence (Config B, civ4_evaluation_summary):")
print(f"  eval run        : {eval_row.mlflow_run_id}")
print(f"  dataset version : {eval_row.eval_dataset_version}")
print(f"  metrics         : faithfulness={eval_row.faithfulness}, "
      f"citation={eval_row.citation_correctness}, correctness={eval_row.answer_correctness}")

# The pinned deployment config must match the evaluated config exactly
assert rag_model.PINNED_CONFIG["config_name"] == eval_row.config_name
assert rag_model.PINNED_CONFIG["top_k"] == eval_row.top_k
assert rag_model.PINNED_CONFIG["prompt_version"] == eval_row.prompt_version
assert rag_model.PINNED_CONFIG["temperature"] == eval_row.temperature
assert rag_model.PINNED_CONFIG["llm_endpoint"] == eval_row.generation_model
assert rag_model.PINNED_CONFIG["embedding_model"] == eval_row.embedding_model
assert rag_model.PINNED_CONFIG["eval_dataset_version"] == eval_row.eval_dataset_version
print("PASS  pinned config matches the evaluated Config B row")

In [0]:
# Idempotency guard — create the registered model once, never duplicate it.
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()
try:
    w.registered_models.get(full_name=REGISTERED_MODEL_NAME)
    print(f"PASS  registered model {REGISTERED_MODEL_NAME} already exists — "
          "this run will append a new VERSION (no duplicate registration)")
    model_already_existed = True
except Exception:
    w.registered_models.create(
        name="civ4_rag_model",
        catalog_name=CATALOG,
        schema_name=SCHEMA,
        comment=(
            "Civ4 RAG assistant (Sprint 6). MLflow PyFunc orchestrating the "
            "Sprint 4 RAG pipeline over Vector Search + Foundation Model APIs. "
            "Pinned config: Sprint 5 Config B (top_k=10, prompt v1, temp 0.1)."
        ),
    )
    print(f"PASS  registered model {REGISTERED_MODEL_NAME} created")
    model_already_existed = False

In [0]:
# Log the model to MLflow — same packaging as notebook 01's validated round trip:
# PythonModel + code_paths (both modules) + signature + input example + pinned deps.
with mlflow.start_run(run_name="civ4_rag_model_configB_deployment") as run:
    run_id = run.info.run_id

    mlflow.log_params({
        "top_k": rag_model.PINNED_CONFIG["top_k"],
        "prompt_version": rag_model.PINNED_CONFIG["prompt_version"],
        "temperature": rag_model.PINNED_CONFIG["temperature"],
        "llm_endpoint": rag_model.PINNED_CONFIG["llm_endpoint"],
        "embedding_model": rag_model.PINNED_CONFIG["embedding_model"],
        "vector_search_index": rag_model.PINNED_CONFIG["index_name"],
        "config_name": rag_model.PINNED_CONFIG["config_name"],
    })
    # Traceability tags — link this deployment to the Sprint 5 evaluation
    mlflow.set_tags({
        "pipeline_stage": "model_serving",
        "sprint": "6",
        "project": "civ4-rag",
        "eval_config_name": eval_row.config_name,
        "eval_mlflow_run_id": eval_row.mlflow_run_id,
        "eval_dataset_version": eval_row.eval_dataset_version,
        "eval_faithfulness": str(eval_row.faithfulness),
        "eval_citation_correctness": str(eval_row.citation_correctness),
        "eval_answer_correctness": str(eval_row.answer_correctness),
        "registered_model_name": REGISTERED_MODEL_NAME,
        "source_repo": "06_model_serving/02_model_registration",
    })

    model_info = mlflow.pyfunc.log_model(
        name="model",
        python_model=rag_model.Civ4RagModel(),
        code_paths=[
            "/Workspace/Repos/endoerner@gmail.com/civ4-rag/06_model_serving/rag_model.py",
            "/Workspace/Repos/endoerner@gmail.com/civ4-rag/04_generation/rag_utils.py",
        ],
        signature=SIG,
        input_example=rag_model.INPUT_EXAMPLE,
        pip_requirements=PIP_REQUIREMENTS,
    )

model_uri = getattr(model_info, "model_uri", None) or f"runs:/{run_id}/model"
print(f"MLflow run       : {run_id}")
print(f"Logged model URI  : {model_uri}")
assert mlflow.search_runs([mlflow.get_experiment_by_name(EXPERIMENT_NAME).experiment_id],
                          filter_string="tags.mlflow.runName = 'civ4_rag_model_configB_deployment'"
                          ).shape[0] >= 1
print("PASS  model logged with params, traceability tags, code_paths, signature, example")

In [0]:
# Register the logged model as a new version in Unity Catalog.
from mlflow.tracking import MlflowClient

client = MlflowClient(registry_uri="databricks-uc")
mv = mlflow.register_model(model_uri, REGISTERED_MODEL_NAME)
print(f"Registered       : {REGISTERED_MODEL_NAME} version {mv.version} (status={mv.status})")
assert mv.name == REGISTERED_MODEL_NAME

# Copy the key traceability tags onto the model version itself,
# so the version is self-describing even outside the MLflow run.
for key, value in {
    "sprint": "6",
    "pipeline_stage": "model_serving",
    "eval_config_name": eval_row.config_name,
    "eval_mlflow_run_id": eval_row.mlflow_run_id,
    "eval_dataset_version": eval_row.eval_dataset_version,
    "source_mlflow_run_id": run_id,
}.items():
    client.set_model_version_tag(REGISTERED_MODEL_NAME, mv.version, key, value)
print("PASS  model version tagged (Sprint 5 traceability + source run)")

In [0]:
# Verify the registration — version metadata, signature, and a zero-cost predict
# through the model loaded FROM THE REGISTRY (empty question, no Vector Search /
# LLM call; full inference is validated in 04_inference_testing).
mv_detail = client.get_model_version(REGISTERED_MODEL_NAME, mv.version)
assert mv_detail.status in ("READY", "REGISTRATION_SUCCESSFUL"), f"version status: {mv_detail.status}"
assert mv_detail.run_id == run_id, "version does not point to this run"
print(f"Version {mv_detail.version} of {mv_detail.name}")
print(f"  source run   : {mv_detail.run_id}")

# Load FROM the registry URI and run a no-cost predict
loaded = mlflow.pyfunc.load_model(f"models:/{REGISTERED_MODEL_NAME}/{mv.version}")
in_sig = loaded.metadata.get_input_schema()
out_sig = loaded.metadata.get_output_schema()
assert "question" in str(in_sig)
assert "answer" in str(out_sig) and "references" in str(out_sig)

resp = loaded.predict(pd.DataFrame({"question": [""]}))
assert "No valid question" in resp[0]["answer"] and resp[0]["references"] == []
print("PASS  model loads from models:/ URI; signature persisted; predict path works (no-cost check)")

In [0]:
# Summary — Sprint 6 registration acceptance.
print("=" * 60)
print("SPRINT 6 NOTEBOOK 02: REGISTRATION COMPLETE")
print("=" * 60)
print(f"""
Registered model : {REGISTERED_MODEL_NAME}
Version          : {mv.version}
MLflow run       : {run_id}
Experiment       : {EXPERIMENT_NAME}
Sprint 5 trace   : config {eval_row.config_name} | eval run {eval_row.mlflow_run_id} | dataset {eval_row.eval_dataset_version}
Packaging        : PyFunc + code_paths (rag_model.py, rag_utils.py) + signature + input example
Dependencies      : {PIP_REQUIREMENTS}
""")
if not model_already_existed:
    assert mv.version == "1", f"expected version 1 on first registration, got {mv.version}"
print("READY for 03_model_serving_deployment")